In [1]:
import torch
import torch.nn as nn
import pandas as pd
import pickle
from torch.nn.utils.rnn import pack_padded_sequence
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load your saved mappings
# (Assuming you saved these as pickles during training)
with open('slug_to_idx.pkl', 'rb') as f:
    slug_to_idx = pickle.load(f)
with open('pid_idx_to_pid.pkl', 'rb') as f:
    pid_idx_to_pid = pickle.load(f)

# Parameters (Must match training exactly)
vocab_size = max(slug_to_idx.values()) + 1
embedding_dim = 256
hidden_dim = 512
num_classes = len(pid_idx_to_pid)

In [7]:
class LSTMRecommender(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(LSTMRecommender, self).__init__()
        # Maps slug indices to dense vectors. padding_idx=0 ensures 0s don't learn.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            dropout=0.2
        )
        
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    # def forward(self, x, lengths):
    #     # x is (batch, seq_len) with Long (integer) values
    #     x = self.embedding(x) 
        
    #     # Pack the sequences for efficiency
    #     packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True)
    #     _, (hn, _) = self.lstm(packed_x)
        
    #     # Use final hidden state of the last LSTM layer
    #     return self.fc(hn[-1])
    def forward(self, x, lengths):
        x = self.embedding(x)
        packed_x = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hn, _) = self.lstm(packed_x)
        
        hidden_vector = hn[-1] # This is the 512-dim vector
        logits = self.fc(hidden_vector) # This is the 1429-dim vector
        
        return logits, hidden_vector

# Initialize and Load Weights
model = LSTMRecommender(vocab_size, embedding_dim, hidden_dim, num_classes).to(device)
model.load_state_dict(torch.load('model(v1).pth'))
model.eval()

C:\Users\user\AppData\Local\Temp\ipykernel_39668\3205521034.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('model(v1).pth'))


LSTMRecommender(
  (embedding): Embedding(728, 256, padding_idx=0)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=512, out_features=1429, bias=True)
)

In [3]:
test_df = pd.read_csv('user_journeys_test.csv')
test_df

,visit_id,slug
0,0000913076882209575,"['manezh-hauck-dreamn-play-naprokat', 'manezh-..."
1,0002268743573412674,"['avtokresla-avtolyulki', 'avtokreslo-britax-r..."
2,0017209440033371282,"['UNKNOWN', 'vesy', 'vesy-sasha-kokon-farla-ba..."
3,0028175944123250496,"['montessori', 'montessori']"
4,0031343418629767038,"['UNKNOWN', 'UNKNOWN', 'avtokresla-avtolyulki'..."
...,...,...
1358,9947592951664175637,"['kolyaski', 'kolyaski', 'kolyaski', 'kolyaski']"
1359,9968262306115660178,"['kacheli-shezlongi', 'kacheli-shezlongi']"
1360,9968912483060354231,"['avtokresla-avtolyulki', 'avtokreslo-britax-r..."
1361,9992384201760487821,"['avtokresla-avtolyulki', 'avtokresla-avtolyul..."


In [22]:
unique_slugs_df = pd.read_csv('unique_slugs_test_df.csv')
unique_slugs_df = unique_slugs_df[unique_slugs_df['pid'] != -1.0]
unique_slugs_df

,slug,pid_is_available,pid
0,manezh-hauck-dreamn-play-naprokat,1,463480358.0
3,avtokreslo-britax-roemer-kidfix-sl-naprokat,1,463480242.0
4,avtokreslo-gruppa-23-15-36-kg-recaro-milano-se...,1,463480342.0
5,avtokreslo-siger-kokon-isofix-naprokat-2,1,463480693.0
6,avtokreslo-britax-romer-kidfix-xp-naprokat,1,495253660.0
...,...,...,...
356,detskiy-basseyn-romana-terra-naprokat,1,463480420.0
357,avtokreslo-ferrari-nania-naprokat,1,463480563.0
358,stulchik-dlya-kormleniya-chicco-new-polly-napr...,1,495403264.0
360,kacheli-shezlong-4moms-mamaroo-40-grafitovyy-m...,1,463480491.0


In [8]:
# Move weights to GPU and normalize
with torch.no_grad():
    # Shape: (num_classes, hidden_dim)
    weights = model.fc.weight.detach()
    # Compute L2 norm for each product vector
    norms = weights.norm(p=2, dim=1, keepdim=True)
    # Normalize: now dot product = cosine similarity
    normalized_weights_gpu = weights / (norms + 1e-10)

In [11]:
results = []
max_window = 10

with torch.no_grad():
    for _, row in test_df.iterrows():
        visit_id = row['visit_id']
        slugs = row['slug']
        
        indices = [slug_to_idx.get(s, 0) for s in slugs][-max_window:]
        x = torch.tensor([indices], dtype=torch.long).to(device)
        lengths = torch.tensor([len(indices)])
        
        # 1. Get BOTH outputs from the model
        logits, hidden_vec = model(x, lengths) 
        
        # --- PART A: Top 3 Direct Predictions ---
        _, top3_indices = torch.topk(logits, k=3, dim=1)
        top3_list = top3_indices[0].cpu().tolist()
        
        # --- PART B: Top 3 by Similarity ---
        # Normalize the hidden vector (1x512)
        query_norm = hidden_vec / (hidden_vec.norm(p=2, dim=1, keepdim=True) + 1e-10)
        
        # Now shapes match: (1x512) @ (512x1429) -> (1x1429)
        sim_scores = torch.matmul(query_norm, normalized_weights_gpu.T) 
        
        _, sim_indices = torch.topk(sim_scores, k=10, dim=1)
        sim_list = sim_indices[0].cpu().tolist()
        
        # Combine and remove duplicates
        final_indices = top3_list.copy()
        for idx in sim_list:
            if idx not in final_indices:
                final_indices.append(idx)
            if len(final_indices) >= 6:
                break
        
        # Map to PIDs
        recommended_pids = [str(pid_idx_to_pid[idx]) for idx in final_indices]
        results.append([visit_id, " ".join(recommended_pids)])

# Save Submission
# submission_df = pd.DataFrame(results, columns=['visit_id', 'product_id'])

In [8]:
# Load your test data
import ast
test_df = pd.read_csv('user_journeys_test.csv') # Ensure 'slug' column is a list of strings
test_df['slug'] = test_df['slug'].apply(ast.literal_eval)
results = []
max_window = 10

with torch.no_grad():
    for _, row in test_df.iterrows():
        visit_id = row['visit_id']
        slugs = row['slug'] # If this is a string, use ast.literal_eval(slugs)
        
        # Convert to indices
        # print(f"Number of slugs found: {len(slugs)}")
        indices = [slug_to_idx.get(s, 0) for s in slugs][-max_window:]
        # print(indices)        
        # Tensors
        x = torch.tensor([indices], dtype=torch.long).to(device)

        lengths = torch.tensor([len(indices)])
        
        # Predict
        outputs = model(x, lengths)
        _, top6_indices = torch.topk(outputs, k=6, dim=1)
        
        # Map back to PIDs
        recommended_pids = [str(pid_idx_to_pid[idx.item()]) for idx in top6_indices[0]]
        
        # Join as space-separated string (standard for many competitions)
        results.append([visit_id, " ".join(recommended_pids)])

In [12]:
results

[['0000913076882209575',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0002268743573412674',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0017209440033371282',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0028175944123250496',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0031343418629767038',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0049025812721671080',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0049122009214721192',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0049949471799959763',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0051677762635990698',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0054376076298156009',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0061180944007759020',
  '495400618 495277192 495400341 495277303 495400448 6501'],
 ['0067449846544086665',
  '495400618 495277192 495400

In [13]:
submissions_df = pd.DataFrame(columns = ['visit_id', 'product_ids'])
submissions_df['visit_id'] = [x[0] for x in results]
submissions_df['product_ids'] = [x[1] for x in results]
submissions_df

,visit_id,product_ids
0,0000913076882209575,495400618 495277192 495400341 495277303 495400...
1,0002268743573412674,495400618 495277192 495400341 495277303 495400...
2,0017209440033371282,495400618 495277192 495400341 495277303 495400...
3,0028175944123250496,495400618 495277192 495400341 495277303 495400...
4,0031343418629767038,495400618 495277192 495400341 495277303 495400...
...,...,...
1358,9947592951664175637,495400618 495277192 495400341 495277303 495400...
1359,9968262306115660178,495400618 495277192 495400341 495277303 495400...
1360,9968912483060354231,495400618 495277192 495400341 495277303 495400...
1361,9992384201760487821,495400618 495277192 495400341 495277303 495400...


In [14]:
(submissions_df['visit_id'] == test_df['visit_id']).sum()

1363

In [15]:
submissions_df['visit_id'].apply(lambda x: x in test_df['visit_id']).sum()

0

In [16]:
test_df['visit_id'].max()

'9994141008995322002'

In [17]:
submissions_df.dtypes

visit_id       object
product_ids    object
dtype: object

In [18]:
test_df.dtypes

visit_id    object
slug        object
dtype: object

In [19]:
submissions_df.to_csv('final_submission.csv', index=False)